In [1]:
from pydantic import BaseModel
import json
import pandas as pd
from pydantic import ValidationError
from pandas import DataFrame
from ollama import generate

In [2]:
class MCQQuestion(BaseModel):
    question: str
    option_a: str
    option_b: str
    option_c: str
    option_d: str
    correct_option: str

In [3]:
def validate_mcq(mcq_json):
    try:
        return MCQQuestion.model_validate_json(mcq_json)
    except ValidationError as e:
        print(f"Validation failed: {e}")
        return None
        


def flatten_and_export_mcq(df: DataFrame, export_filename: str, mcq_column_name: str):
    result_df = df[['id']].copy()
    
    result_df['question'] = df[mcq_column_name].apply(lambda x: x.question if x else "")
    result_df['option_a'] = df[mcq_column_name].apply(lambda x: x.option_a if x else "")
    result_df['option_b'] = df[mcq_column_name].apply(lambda x: x.option_b if x else "")
    result_df['option_c'] = df[mcq_column_name].apply(lambda x: x.option_c if x else "")
    result_df['option_d'] = df[mcq_column_name].apply(lambda x: x.option_d if x else "")
    result_df['correct_option'] = df[mcq_column_name].apply(lambda x: x.correct_option if x else "")
    
    result_df.to_csv(export_filename, index=False)

In [4]:

def generate_mcq(content, model_name, temperature):
    prompt = f"""
    À partir du contenu éducatif suivant, générez deux questions à choix multiple avec quatre options de réponse dont une seule est correcte.
    La question doit évaluer la compréhension des idées principales, et les options doivent être claires, informatives et pertinentes.
    Assurez-vous que les distracteurs (options incorrectes) suivent une interprétation logique mais incorrecte, basée sur des idées reçues ou des incompréhensions courantes du sujet.  
    Les options de réponse doivent être aussi courtes que possible.

    **Contenu éducatif**

    {content}
    """
    
    generate_params = {
        'model': model_name,
        'options': {'temperature': temperature, 'num_ctx': 8192, 'top_p': 1}, 
        'prompt': prompt,
        'format': MCQQuestion.model_json_schema()
    }
    
    # Get a response
    response = generate(**generate_params)
    
    return response['response']

In [5]:
df = pd.read_csv("../data/lisa_sheets.csv")

In [6]:
file_path = "../data/train_test_split/test_folders.json"

In [7]:
with open(file_path, "r", encoding="utf-8") as file:
    test_folders = json.load(file)

In [8]:
df_test = df[df.folder.isin(test_folders)]

In [ ]:
%%time
df_test['generated_llama3_1_8b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="llama3.1:8b", temperature=0.1)
)

df_test["llama3_1_8b"] = df_test['generated_llama3_1_8b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/llama3_1_8b.csv', 'llama3_1_8b')

In [ ]:
%%time
df_test['generated_openbiollm_8b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="koesn/llama3-openbiollm-8b:latest", temperature=0.1)
)

df_test["openbiollm_8b"] = df_test['generated_openbiollm_8b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/openbiollm_8b.csv', 'openbiollm_8b')

In [ ]:
%%time
df_test['generated_gemma2_9b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="gemma2:9b", temperature=0.1)
)

df_test["gemma2_9b"] = df_test['generated_gemma2_9b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/gemma2_9b.csv', 'gemma2_9b')

In [ ]:
%%time
df_test['generated_medGemma_4b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="alibayram/medgemma:4b", temperature=0.1)
)

df_test["medGemma_4b"] = df_test['generated_medGemma_4b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/medGemma_4b.csv', 'medGemma_4b')

In [9]:
%%time
df_test['generated_medGemma_27b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="alibayram/medgemma:27b", temperature=0.1)
)

df_test["medGemma_27b"] = df_test['generated_medGemma_27b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/medGemma_27b.csv', 'medGemma_27b')

CPU times: user 4.18 s, sys: 315 ms, total: 4.5 s
Wall time: 1h 52min 4s


<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
# One-liner version
#df_test[df_test['validated_mcq_0.5'].isna()].index.tolist()

In [ ]:
empty_mcq = MCQQuestion(
    question="",
    option_a="",
    option_b="",
    option_c="",
    option_d="",
    correct_option=""
)

# Set the value at the specified index
#df_test.at[3742, 'validated_mcq_0.5'] = empty_mcq